In [19]:
# Load
import pandas as pd
import numpy as np

train = pd.read_csv('../data/splits/train.csv') 
val = pd.read_csv('../data/splits/val.csv') 
test = pd.read_csv('../data/splits/test.csv') 

print(train.shape)
train.head()

(32411190, 29)


,provider_code,pickup_timestamp,dropoff_timestamp,rider_count,distance_miles,rate_class_id,offline_record_flag,origin_loc_id,dest_loc_id,fare_settlement_method,...,congestion_relief_fee,rider_count_missing,calculated_charge,rider_count_was_zero,pickup_borough,pickup_zone,pickup_service_zone,dropoff_borough,dropoff_zone,dropoff_service_zone
0,2,2025-06-26 22:08:31,2025-06-26 22:27:38,1.0,2.37,1.0,N,230,125,0,...,0.75,True,-2.50,False,Manhattan,Times Sq/Theatre District,Yellow Zone,Manhattan,Hudson Sq,Yellow Zone
1,2,2025-11-28 16:13:28,2025-11-28 16:32:37,1.0,3.51,1.0,N,4,100,0,...,0.75,True,0.04,False,Manhattan,Alphabet City,Yellow Zone,Manhattan,Garment District,Yellow Zone
2,2,2025-11-13 13:23:43,2025-11-13 13:31:33,1.0,1.05,1.0,N,170,107,4,...,0.75,False,14.05,False,Manhattan,Murray Hill,Yellow Zone,Manhattan,Gramercy,Yellow Zone
3,2,2025-10-17 18:04:20,2025-10-17 18:11:58,2.0,0.53,1.0,N,161,161,4,...,0.75,False,15.15,False,Manhattan,Midtown Center,Yellow Zone,Manhattan,Midtown Center,Yellow Zone
4,2,2025-07-08 18:29:41,2025-07-08 18:31:51,2.0,0.40,1.0,N,170,170,1,...,0.75,False,13.98,False,Manhattan,Murray Hill,Yellow Zone,Manhattan,Murray Hill,Yellow Zone


In [20]:

train['pickup_timestamp'] = pd.to_datetime(train['pickup_timestamp'])
train['pickup_hour'] = train['pickup_timestamp'].dt.hour
train['pickup_dayofweek'] = train['pickup_timestamp'].dt.dayofweek  # 0=Mon
train['pickup_month'] = train['pickup_timestamp'].dt.month
train['pickup_day'] = train['pickup_timestamp'].dt.day
train['is_weekend'] = train['pickup_dayofweek'].isin([5, 6]).astype(int)

# Rush hour flag (typical NYC AM/PM rush: 7-10am, 4-7pm)
train['is_rush_hour'] = train['pickup_hour'].isin([7,8,9,16,17,18]).astype(int)

# Night flag (late night trips often priced/behave differently)
train['is_night'] = train['pickup_hour'].isin([22,23,0,1,2,3,4,5]).astype(int)

train[['pickup_timestamp','pickup_hour','pickup_dayofweek','is_weekend','is_rush_hour','is_night']].head()

,pickup_timestamp,pickup_hour,pickup_dayofweek,is_weekend,is_rush_hour,is_night
0,2025-06-26 22:08:31,22,3,0,0,1
1,2025-11-28 16:13:28,16,4,0,1,0
2,2025-11-13 13:23:43,13,3,0,0,0
3,2025-10-17 18:04:20,18,4,0,1,0
4,2025-07-08 18:29:41,18,1,0,1,0


In [21]:
# Apply same transformations to val and test sets
for df in [val, test]:
    df['pickup_timestamp'] = pd.to_datetime(df['pickup_timestamp'])
    df['pickup_hour'] = df['pickup_timestamp'].dt.hour
    df['pickup_dayofweek'] = df['pickup_timestamp'].dt.dayofweek
    df['pickup_month'] = df['pickup_timestamp'].dt.month
    df['pickup_day'] = df['pickup_timestamp'].dt.day
    df['is_weekend'] = df['pickup_dayofweek'].isin([5, 6]).astype(int)
    df['is_rush_hour'] = df['pickup_hour'].isin([7,8,9,16,17,18]).astype(int)
    df['is_night'] = df['pickup_hour'].isin([22,23,0,1,2,3,4,5]).astype(int)

In [22]:
# Airport pickup/dropoff flags (from rate_class_id per data dictionary)
# rate_class_id: 2=JFK, 3=Newark
for df in [train, val, test]:
    df['is_airport_rate'] = df['rate_class_id'].isin([2, 3]).astype(int)
    df['is_negotiated_fare'] = (df['rate_class_id'] == 5).astype(int)
    df['is_group_ride'] = (df['rate_class_id'] == 6).astype(int)

train[['rate_class_id','is_airport_rate','is_negotiated_fare','is_group_ride']].head()

,rate_class_id,is_airport_rate,is_negotiated_fare,is_group_ride
0,1.0,0,0,0
1,1.0,0,0,0
2,1.0,0,0,0
3,1.0,0,0,0
4,1.0,0,0,0


In [23]:
# Same-zone trip flag (pickup == dropoff, might indicate short local trip or round trip)
for df in [train, val, test]:
    df['same_zone_trip'] = (df['origin_loc_id'] == df['dest_loc_id']).astype(int)

In [24]:
# Cell: Check cardinality first — determines encoding strategy
print("Unique origin zones:", train['origin_loc_id'].nunique())
print("Unique dest zones:", train['dest_loc_id'].nunique())
print("Unique pickup_borough:", train['pickup_borough'].nunique())
print("Unique rate_class_id:", train['rate_class_id'].nunique())

Unique origin zones: 262
Unique dest zones: 262
Unique pickup_borough: 7
Unique rate_class_id: 7


In [25]:
# Target encoding for high-cardinality zone features (origin/dest zone)
# Using mean base_fare per zone, computed ONLY on train set to avoid leakage
def target_encode(train_df, val_df, test_df, col, target='base_fare'):
    means = train_df.groupby(col)[target].mean()
    global_mean = train_df[target].mean()
    
    train_df[col + '_target_enc'] = train_df[col].map(means)
    val_df[col + '_target_enc'] = val_df[col].map(means).fillna(global_mean)
    test_df[col + '_target_enc'] = test_df[col].map(means).fillna(global_mean)
    return train_df, val_df, test_df

train, val, test = target_encode(train, val, test, 'origin_loc_id')
train, val, test = target_encode(train, val, test, 'dest_loc_id')

In [26]:
# One-hot encode low-cardinality categoricals (borough, rate_class)
categorical_low_card = ['pickup_borough', 'dropoff_borough', 'provider_code']

train = pd.get_dummies(train, columns=categorical_low_card, drop_first=True)
val = pd.get_dummies(val, columns=categorical_low_card, drop_first=True)
test = pd.get_dummies(test, columns=categorical_low_card, drop_first=True)

# Align columns across splits (in case val/test missing some dummy categories)
train, val = train.align(val, join='left', axis=1, fill_value=0)
train, test = train.align(test, join='left', axis=1, fill_value=0)

In [27]:
# Assemble final feature list
fare_model_features = [
    'pickup_hour', 'pickup_dayofweek', 'pickup_month', 'is_weekend',
    'is_rush_hour', 'is_night', 'rider_count',
    'is_airport_rate', 'is_negotiated_fare', 'is_group_ride', 'same_zone_trip',
    'origin_loc_id_target_enc', 'dest_loc_id_target_enc', 'distance_miles'
] + [col for col in train.columns if col.startswith('pickup_borough_') 
     or col.startswith('dropoff_borough_') or col.startswith('provider_code_')]

target = 'base_fare'

X_train = train[fare_model_features]
y_train = train[target]

X_val = val[fare_model_features]
y_val = val[target]

X_test = test[fare_model_features]
y_test = test[target]

print("Feature count:", len(fare_model_features))
print("X_train shape:", X_train.shape)

Feature count: 29
X_train shape: (32411190, 29)


In [28]:
# Cell: Save feature-engineered datasets
train.to_csv('train_features.csv', index=False)
val.to_csv('val_features.csv', index=False)
test.to_csv('test_features.csv', index=False)

Linear Regression


In [29]:
# Baseline - Linear Regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict on validation set
y_val_pred_lr = lr.predict(X_val)

# Evaluate
rmse_lr = np.sqrt(mean_squared_error(y_val, y_val_pred_lr))
mae_lr = mean_absolute_error(y_val, y_val_pred_lr)
r2_lr = r2_score(y_val, y_val_pred_lr)

print("=== Linear Regression Baseline ===")
print(f"RMSE: {rmse_lr:.3f}")
print(f"MAE:  {mae_lr:.3f}")
print(f"R²:   {r2_lr:.3f}")

=== Linear Regression Baseline ===
RMSE: 11.216
MAE:  6.899
R²:   0.612


XGBoost

In [30]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)
y_val_pred_xgb = xgb.predict(X_val)

rmse_xgb = np.sqrt(mean_squared_error(y_val, y_val_pred_xgb))
mae_xgb = mean_absolute_error(y_val, y_val_pred_xgb)
r2_xgb = r2_score(y_val, y_val_pred_xgb)

print("=== XGBoost (default) ===")
print(f"RMSE: {rmse_xgb:.3f}")
print(f"MAE:  {mae_xgb:.3f}")
print(f"R²:   {r2_xgb:.3f}")

=== XGBoost (default) ===
RMSE: 13.779
MAE:  3.948
R²:   0.415


In [31]:
import pickle

with open("xgboost_base_fare_model.pkl", "wb") as file:
    pickle.dump(xgb, file)

print("Model saved successfully!")

Model saved successfully!
